# OpenAI API 最新機能体験：Web検索とCSV分析

Google Colaboratoryで、Responses APIの次の機能を試します。

1. 通常のテキスト生成（Chat Completions）
2. API内蔵のWeb検索
3. CSVをAPIへアップロード
4. Code InterpreterがPythonで集計・グラフ作成
5. AIが生成したCSV・画像をColabへ保存
6. Images APIでイラストを生成してPNG保存
7. Responses APIの image_generation で生成し、続けて編集する

> **注意**：APIはChatGPTの月額契約とは別の従量課金です。患者名、患者IDなどの個人情報は入力しないでください。このNotebookでは `store=False` を指定します。

## 公式ドキュメント

この資料を見ながら、OpenAIの原点のページも開いて学習してください。

- [API keys](https://platform.openai.com/api-keys)
- [Developer quickstart](https://developers.openai.com/api/docs/quickstart)
- [Chat Completions](https://platform.openai.com/docs/api-reference/chat/create)（文章を返すだけの古い書き方）
- [Text generation](https://developers.openai.com/api/docs/guides/text)（Responses APIの基本）
- [Web search](https://developers.openai.com/api/docs/guides/tools-web-search)
- [File inputs](https://developers.openai.com/api/docs/guides/file-inputs)
- [Code Interpreter](https://developers.openai.com/api/docs/guides/tools-code-interpreter)
- [Image generation](https://developers.openai.com/api/docs/guides/image-generation)
- [Image generation tool](https://developers.openai.com/api/docs/guides/tools-image-generation)
- [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state)（`previous_response_id`）

## 0. 事前準備

1. OpenAI PlatformでAPIキーを発行します。
2. Colab左側の鍵アイコン「シークレット」を開きます。
3. 名前を `OPENAI_API_KEY` としてAPIキーを登録します。
4. 「ノートブックからのアクセス」をONにします。

APIキーをコードへ直接貼り付けないでください。

In [ ]:
# pandasとmatplotlibはColab標準版を使う
!pip install -q --upgrade openai

In [ ]:
from pathlib import Path
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Colabのシークレットに OPENAI_API_KEY を登録してください。")

client = OpenAI(api_key=api_key)

# 費用と性能のバランスを重視。利用できない場合は、自分のプロジェクトで
# 利用可能なモデル名に変更してください。
MODEL = "gpt-5.6-terra"
print("準備完了：", MODEL)

## 1. 普通のAI回答

まずは今も使える Chat Completions で、文章を返すだけを試します。
次の Web 検索からは Responses API（`client.responses.create`）に切り替えます。

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "医療機関が生成AIを利用するときの注意点を3つ、簡潔に説明してください。",
        }
    ],
)

print(response.choices[0].message.content)

## 2. API内蔵のWeb検索

ここから Responses API です。`tools` に `web_search` を渡すと、モデルが検索し、最新情報を出典付きで整理します。

In [ ]:
web_response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "web_search",
            "filters": {
                "allowed_domains": [
                    "mhlw.go.jp",
                    "digital.go.jp",
                ]
            },
        }
    ],
    tool_choice="required",
    include=["web_search_call.action.sources"],
    input=(
        "厚生労働省とデジタル庁の公開情報を検索し、医療機関に関係する"
        "生成AIまたは医療DXの最近の動向を3つ整理してください。"
        "各項目を『概要』『医療機関への影響』『出典』に分けてください。"
    ),
    store=False,
)

print(web_response.output_text)

### 検索処理の中身を見る

レスポンス全体には、検索語や参照元など、画面表示以外の構造化データも含まれます。

In [ ]:
for item in web_response.output:
    if item.type == "web_search_call":
        print(item.model_dump_json(indent=2))

## 3. 分析用のサンプルCSVを作る

実在する患者・職員・病院のデータではありません。勉強会用の架空データです。

In [ ]:
import pandas as pd

sample_data = [
    ["2026-01", "内科", 820, 112000000, 68000000],
    ["2026-01", "整形外科", 640, 98000000, 61000000],
    ["2026-01", "小児科", 410, 35000000, 24000000],
    ["2026-02", "内科", 790, 108000000, 67000000],
    ["2026-02", "整形外科", 675, 104000000, 63000000],
    ["2026-02", "小児科", 460, 39000000, 25000000],
    ["2026-03", "内科", 850, 118000000, 70000000],
    ["2026-03", "整形外科", 710, 109000000, 65000000],
    ["2026-03", "小児科", 390, 33000000, 23500000],
    ["2026-04", "内科", 875, 121000000, 71000000],
    ["2026-04", "整形外科", 690, 106000000, 64500000],
    ["2026-04", "小児科", 430, 37000000, 24500000],
]

columns = ["月", "診療科", "患者数", "売上", "費用"]
df = pd.DataFrame(sample_data, columns=columns)
df["利益"] = df["売上"] - df["費用"]

SAMPLE_CSV_PATH = Path("hospital_sales_sample.csv")
df.to_csv(SAMPLE_CSV_PATH, index=False, encoding="utf-8-sig")
display(df)
print("作成：", SAMPLE_CSV_PATH)

## 4. 使用するCSVを選ぶ

初期状態では上のサンプルCSVを使います。自分のCSVを使う場合だけ `USE_OWN_CSV = True` に変更してください。個人情報・機密情報はアップロードしないでください。

In [ ]:
from google.colab import files

USE_OWN_CSV = False

if USE_OWN_CSV:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("CSVが選択されませんでした。")
    CSV_PATH = Path(next(iter(uploaded.keys())))
else:
    CSV_PATH = SAMPLE_CSV_PATH

if CSV_PATH.suffix.lower() != ".csv":
    raise ValueError("この演習ではCSVファイルを選択してください。")

print("分析対象：", CSV_PATH)

## 5. CSVをOpenAI APIへアップロードする

モデル入力として使うファイルなので、`purpose="user_data"` を指定します。

In [ ]:
with open(CSV_PATH, "rb") as f:
    api_file = client.files.create(
        file=f,
        purpose="user_data",
    )

print("アップロード完了")
print("File ID:", api_file.id)

## 6. Code InterpreterでCSVを分析する

モデルにCSVを読ませるだけでなく、隔離された実行環境でPythonを実行させます。集計結果CSVとグラフPNGも作らせます。

In [ ]:
analysis_prompt = """
添付CSVをpython toolで読み込み、必ず実際にPythonコードを実行して分析してください。

実施事項：
1. 列名、行数、欠損値、データ型を確認する
2. データ全体の特徴を日本語で説明する
3. 数値列を適切に集計する
4. 月や部門・診療科に相当する列があれば、推移と比較を分析する
5. 注目すべき変化を3つ挙げる
6. 集計結果を /mnt/data/analysis_result.csv にUTF-8で保存する
7. 日本語フォントの文字化けを避けたグラフを作成し、
   /mnt/data/analysis_chart.png に保存する

データから確認できる事実と、推測・仮説を明確に分けてください。
"""

analysis_response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "code_interpreter",
            "container": {
                "type": "auto",
                "file_ids": [api_file.id],
            },
        }
    ],
    tool_choice="required",
    input=analysis_prompt,
    store=False,
)

print(analysis_response.output_text)

### AIが実行したPythonコードを確認する

回答文だけでなく、モデルがどのようなコードを実行したか確認します。

In [ ]:
container_id = None

for item in analysis_response.output:
    if item.type == "code_interpreter_call":
        container_id = item.container_id
        print(item.model_dump_json(indent=2))

if not container_id:
    raise RuntimeError("Code Interpreterのcontainer_idを取得できませんでした。")

print("Container ID:", container_id)

## 7. AIが作成したファイルをColabへ保存する

コンテナ内のファイル一覧を取得し、入力ファイル以外を `openai_outputs` フォルダへ保存します。コンテナは最終使用から20分で期限切れになるため、必要なファイルはすぐ取得します。

In [ ]:
OUTPUT_DIR = Path("openai_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

container_files = list(client.containers.files.list(container_id=container_id))
downloaded_paths = []

for container_file in container_files:
    filename = Path(container_file.path or container_file.id).name

    # 最初から入っている入力CSVは除外し、AIが生成したファイルを保存します。
    if filename == CSV_PATH.name:
        continue

    destination = OUTPUT_DIR / filename
    content = client.containers.files.content.retrieve(
        container_id=container_id,
        file_id=container_file.id,
    )
    content.write_to_file(destination)
    downloaded_paths.append(destination)
    print("保存：", destination)

if not downloaded_paths:
    print("生成ファイルが見つかりませんでした。モデルの回答と実行ログを確認してください。")

### 生成されたグラフと集計表を表示する

In [ ]:
from IPython.display import display, Image

for path in downloaded_paths:
    if path.suffix.lower() == ".png":
        display(Image(filename=str(path)))
    elif path.suffix.lower() == ".csv":
        print("\n", path.name)
        display(pd.read_csv(path))

### 必要なファイルをPCへダウンロードする

In [ ]:
# ダウンロードしたいファイルについて、次の行のコメントを外してください。
# files.download(str(OUTPUT_DIR / "analysis_result.csv"))
# files.download(str(OUTPUT_DIR / "analysis_chart.png"))

## 8. Images APIで画像を生成する

Responses APIとは別に、Images APIでイラストを生成できます。品質重視は `gpt-image-2.5-sunburst`、速度重視は `gpt-image-2.5-flare` です。

In [ ]:
import base64
from pathlib import Path
from IPython.display import display, Image

# 高品質・編集精度重視
IMAGE_MODEL = "gpt-image-2.5-sunburst"

# 速度重視ならこちら
# IMAGE_MODEL = "gpt-image-2.5-flare"

prompt = """
明るく清潔感のある日本の地域病院で、
医療従事者とAIロボットが協力してデータ分析をしている様子。

現代的なフラットイラスト。
青と白を基調とした配色。
勉強会のスライドに使える構図。
画像内に文字は入れない。
"""

result = client.images.generate(
    model=IMAGE_MODEL,
    prompt=prompt,
    size="1024x1024",
    quality="high",
)

# Base64データを画像へ変換
image_base64 = result.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

# PNGとして保存
IMAGE_PATH = Path("generated_image.png")
IMAGE_PATH.write_bytes(image_bytes)

# Notebook上に表示
display(Image(filename=str(IMAGE_PATH)))

print("保存しました：", IMAGE_PATH)

## 9. Responses APIで画像を生成し、続けて編集する

`client.images.generate` ではなく、Responses APIの `image_generation` ツールでも作れます。続きの変更指示を出すため、ここだけ `store=True` にし、2回目は `previous_response_id` で1回目を参照します。

In [ ]:
import base64
from IPython.display import display, Image

# 1回目：画像生成
response1 = client.responses.create(
    model="gpt-5.6-luna",
    input="""
    日本の地域病院で、医療従事者とAIロボットが
    データ分析している明るいイラストを作ってください。
    画像内に文字は入れないでください。
    """,
    tools=[
        {
            "type": "image_generation",
            "model": "gpt-image-2.5-sunburst",
            "quality": "high",
        }
    ],
    store=True,
)

image_data1 = [
    item.result
    for item in response1.output
    if item.type == "image_generation_call"
][0]

with open("image_v1.png", "wb") as f:
    f.write(base64.b64decode(image_data1))

display(Image(filename="image_v1.png"))

In [ ]:
# 2回目：「さっきの画像」への変更指示
response2 = client.responses.create(
    model="gpt-5.6-luna",
    previous_response_id=response1.id,
    input="""
    さっきの画像の構図と人物を維持したまま、
    AIロボットを小さくし、背景にグラフ画面を追加してください。
    """,
    tools=[
        {
            "type": "image_generation",
            "model": "gpt-image-2.5-sunburst",
            "quality": "high",
        }
    ],
    store=True,
)

image_data2 = [
    item.result
    for item in response2.output
    if item.type == "image_generation_call"
][0]

with open("image_v2.png", "wb") as f:
    f.write(base64.b64decode(image_data2))

display(Image(filename="image_v2.png"))

## 10. 自由課題

`analysis_prompt`を次のように変更して試してみましょう。

- 売上が最も伸びた診療科を特定する
- 患者1人当たり売上を計算する
- 前月比を計算して異常値を探す
- 経営会議向けに200文字で要約する
- Excelファイルを作成するよう指示する

モデルへ『Pythonコードを書いて』ではなく『python toolで実際に実行して』と伝えるのがポイントです。

## 11. 後片付け（任意）

演習が終わったら、APIへアップロードした元CSVを削除できます。実行後は同じFile IDを再利用できません。

In [ ]:
DELETE_UPLOADED_FILE = False

if DELETE_UPLOADED_FILE:
    client.files.delete(api_file.id)
    print("API上の元CSVを削除しました。")
else:
    print("削除していません。削除する場合は True に変更してください。")

## まとめ

このNotebookでは、OpenAI APIを単なる文章生成としてではなく、次のように利用しました。

- 最新情報を検索する
- ファイルを受け取る
- Pythonコードを自ら作成して実行する
- 集計表やグラフを成果物として返す
- Images APIでイラストを生成して保存する
- Responses APIで生成し、previous_response_id で続き編集する

実業務では、個人情報、機密情報、出力の検証、利用上限、エラー処理を追加してください。